In [1]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Load Function 5 data
X = np.load("function5/initial_inputs.npy")
Y = np.load("function5/initial_outputs.npy")
print("X shape:", X.shape)
print("Y shape:", Y.shape)


print(X)


print(Y)

X shape: (20, 4)
Y shape: (20,)
[[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]]
[6.44434399e+01 1.83013796e

In [2]:
best_index = np.argmax(Y)

print("Best index:", best_index)
print("Best current x:", X[best_index])
print("Best current y:", Y[best_index])
print("Best current portal format:", "-".join(f"{v:.6f}" for v in X[best_index]))

kernel = C(1.0) * RBF(length_scale=0.2)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    random_state=42
)

gp.fit(X, Y)

rng = np.random.default_rng(42)
candidates = rng.uniform(0, 1, size=(10000, X.shape[1]))

mean, std = gp.predict(candidates, return_std=True)

kappa = 2.5
ucb = mean + kappa * std

best_ucb_index = np.argmax(ucb)
query = candidates[best_ucb_index]

print("Suggested query:", query)
print("Portal format:", "-".join(f"{v:.6f}" for v in query))
print("Predicted mean:", mean[best_ucb_index])
print("Predicted std:", std[best_ucb_index])
print("UCB score:", ucb[best_ucb_index])

Best index: 15
Best current x: [0.22418902 0.84648049 0.87948418 0.87851568]
Best current y: 1088.8596181962705
Best current portal format: 0.224189-0.846480-0.879484-0.878516
Suggested query: [0.37751058 0.87159411 0.98865228 0.96779518]
Portal format: 0.377511-0.871594-0.988652-0.967795
Predicted mean: 1101.963124950801
Predicted std: 144.92813678690135
UCB score: 1464.2834669180545


In [3]:
import numpy as np
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Load original Function 5 data
X = np.load("function5/initial_inputs.npy")
Y = np.load("function5/initial_outputs.npy")

# Add Week 1 query and output
week1_x = np.array([[0.377511, 0.871594, 0.988652, 0.967795]])
week1_y = np.array([2687.686511102482])

X = np.vstack([X, week1_x])
Y = np.append(Y, week1_y)

print("Updated X shape:", X.shape)
print("Updated Y shape:", Y.shape)

print("Best current x:", X[np.argmax(Y)])
print("Best current y:", np.max(Y))

# Fit GP surrogate model
kernel = C(1.0) * RBF(length_scale=0.2)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    random_state=42
)

gp.fit(X, Y)

# Generate candidate points
rng = np.random.default_rng(42)
candidates = rng.uniform(0, 1, size=(10000, X.shape[1]))

# Predict mean and uncertainty
mean, std = gp.predict(candidates, return_std=True)

# Expected Improvement acquisition
y_best = np.max(Y)
std_safe = std + 1e-12

improvement = mean - y_best
z = improvement / std_safe
ei = improvement * norm.cdf(z) + std_safe * norm.pdf(z)

best_ei_index = np.argmax(ei)
query = candidates[best_ei_index]

print("Acquisition used: Expected Improvement")
print("Suggested query:", query)
print("Portal format:", "-".join(f"{v:.6f}" for v in query))
print("Predicted mean:", mean[best_ei_index])
print("Predicted std:", std[best_ei_index])
print("EI score:", ei[best_ei_index])

Updated X shape: (21, 4)
Updated Y shape: (21,)
Best current x: [0.377511 0.871594 0.988652 0.967795]
Best current y: 2687.686511102482
Acquisition used: Expected Improvement
Suggested query: [0.39834821 0.89618367 0.9269388  0.97283595]
Portal format: 0.398348-0.896184-0.926939-0.972836
Predicted mean: 2510.496129644985
Predicted std: 201.32338549366048
EI score: 20.966139469295584


In [4]:
import numpy as np
from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C


# For Function 5, Week 1 and Week 2 both produced very high outputs.
# Week 1 remains the best observed value, while Week 2 confirmed that the surrounding region is strong.
# Because of this, I use a mostly exploitative local EI strategy around the current best point,
# with a small UCB component to avoid choosing a point with no uncertainty.


# -----------------------------
# Load original Function 5 data
# -----------------------------
X = np.load("function5/initial_inputs.npy")
Y = np.load("function5/initial_outputs.npy")


# -----------------------------
# Add Week 1 and Week 2 results
# -----------------------------
week1_x = np.array([[0.377511, 0.871594, 0.988652, 0.967795]])
week1_y = np.array([2687.686511102482])

week2_x = np.array([[0.398348, 0.896184, 0.926939, 0.972836]])
week2_y = np.array([2360.0402820167205])

X = np.vstack([X, week1_x, week2_x])
Y = np.append(Y, [week1_y[0], week2_y[0]])


print("Updated X shape:", X.shape)
print("Updated Y shape:", Y.shape)

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("Best current x:", best_x)
print("Best current y:", best_y)


# -----------------------------
# Fit GP surrogate model
# -----------------------------
kernel = (
    C(1.0, (1e-3, 1e3))
    * Matern(length_scale=np.ones(X.shape[1]) * 0.2, length_scale_bounds=(1e-2, 1.0), nu=2.5)
    + WhiteKernel(noise_level=1e-8, noise_level_bounds=(1e-10, 1e-3))
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

gp.fit(X, Y)

print("Fitted kernel:", gp.kernel_)


# -----------------------------
# Generate candidate points
# -----------------------------
rng = np.random.default_rng(42)
dim = X.shape[1]

# Function 5 already has a very strong region, so generate mostly local candidates.
local_candidates = rng.normal(loc=best_x, scale=0.05, size=(30000, dim))
local_candidates = np.clip(local_candidates, 0, 1)

# Add some candidates around the Week 2 point because it also performed strongly.
week2_candidates = rng.normal(loc=week2_x[0], scale=0.05, size=(10000, dim))
week2_candidates = np.clip(week2_candidates, 0, 1)

# Small global pool to avoid total overfitting.
global_candidates = rng.uniform(0, 1, size=(5000, dim))

candidates = np.vstack([local_candidates, week2_candidates, global_candidates])


# -----------------------------
# Predict mean and uncertainty
# -----------------------------
mean, std = gp.predict(candidates, return_std=True)

y_best = np.max(Y)
std_safe = std + 1e-12


# -----------------------------
# Expected Improvement
# -----------------------------
improvement = mean - y_best
z = improvement / std_safe
ei = improvement * norm.cdf(z) + std_safe * norm.pdf(z)


# -----------------------------
# Upper Confidence Bound
# -----------------------------
# Lower kappa because Function 5 has a strong known region and should lean exploitative.
kappa = 0.8
ucb = mean + kappa * std


# -----------------------------
# Strict filtered local hybrid EI-UCB
# -----------------------------
# Only consider candidates with strong predicted performance.
# Because Function 5 outputs are large, we use a relative threshold.
mean_filter = mean >= np.percentile(mean, 85)

filtered_candidates = candidates[mean_filter]
filtered_mean = mean[mean_filter]
filtered_std = std[mean_filter]
filtered_ei = ei[mean_filter]
filtered_ucb = ucb[mean_filter]

ei_norm = (filtered_ei - np.min(filtered_ei)) / (np.max(filtered_ei) - np.min(filtered_ei) + 1e-12)
ucb_norm = (filtered_ucb - np.min(filtered_ucb)) / (np.max(filtered_ucb) - np.min(filtered_ucb) + 1e-12)

# EI is dominant because the region is already promising.
hybrid_score = 0.85 * ei_norm + 0.15 * ucb_norm

best_hybrid_index = np.argmax(hybrid_score)
query = filtered_candidates[best_hybrid_index]


# -----------------------------
# Output results
# -----------------------------
print("Acquisition used: Exploitative filtered local Hybrid EI + UCB")
print("Number of candidates passing filter:", np.sum(mean_filter))
print("Suggested query:", query)

print("Portal format with hyphens:")
print("-".join(f"{v:.6f}" for v in query))

print("Portal format with x labels:")
print(",".join(f"x{i+1}:{v:.6f}" for i, v in enumerate(query)))

print("Predicted mean:", filtered_mean[best_hybrid_index])
print("Predicted std:", filtered_std[best_hybrid_index])
print("EI score:", filtered_ei[best_hybrid_index])
print("UCB score:", filtered_ucb[best_hybrid_index])
print("Hybrid score:", hybrid_score[best_hybrid_index])

Updated X shape: (22, 4)
Updated Y shape: (22,)
Best current x: [0.377511 0.871594 0.988652 0.967795]
Best current y: 2687.686511102482
Fitted kernel: 0.923**2 * Matern(length_scale=[0.164, 1, 0.566, 1], nu=2.5) + WhiteKernel(noise_level=0.001)
Acquisition used: Exploitative filtered local Hybrid EI + UCB
Number of candidates passing filter: 6750
Suggested query: [0.35918062 0.92823272 1.         1.        ]
Portal format with hyphens:
0.359181-0.928233-1.000000-1.000000
Portal format with x labels:
x1:0.359181,x2:0.928233,x3:1.000000,x4:1.000000
Predicted mean: 2737.136551645922
Predicted std: 86.98414184759545
EI score: 64.88794371532927
UCB score: 2806.723865123998
Hybrid score: 0.9940831559082288


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.001. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [6]:
# week 4 

week3_x = np.array([[0.359181, 0.928233, 1.000000, 1.000000]])
week3_y = np.array([3650.138639563766])

X = np.vstack([X, week1_x, week2_x, week3_x])
Y = np.append(Y, [week1_y[0], week2_y[0], week3_y[0]])

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("Shape:", X.shape, Y.shape)
print("Current best x:", best_x)
print("Current best y:", best_y)

Shape: (28, 4) (28,)
Current best x: [0.359181 0.928233 1.       1.      ]
Current best y: 3650.138639563766


In [7]:
Y_mean = Y.mean()
Y_std = Y.std()

Y_scaled = (Y - Y_mean) / Y_std

print("Output mean:", Y_mean)
print("Output std:", Y_std)

Output mean: 909.6033990023358
Output std: 1246.7757295787726


In [8]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel as C,
    Matern,
    WhiteKernel
)

kernel = (
    C(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.full(4, 0.2),
        length_scale_bounds=(1e-2, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-8,
        noise_level_bounds=(1e-10, 1e-3)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=False,
    n_restarts_optimizer=25,
    random_state=42
)

gp.fit(X, Y_scaled)

print("Fitted kernel:", gp.kernel_)

/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 13 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 17 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 16 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/

Fitted kernel: 0.95**2 * Matern(length_scale=[2, 0.299, 0.377, 2], nu=2.5) + WhiteKernel(noise_level=1e-10)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 14 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
 

In [9]:
trust_radius = np.array([
    0.12,  # x1
    0.08,  # x2
    0.03,  # x3
    0.03   # x4
])

lower_bounds = np.maximum(0, best_x - trust_radius)
upper_bounds = np.minimum(1, best_x + trust_radius)

bounds = list(zip(lower_bounds, upper_bounds))

print("Lower bounds:", lower_bounds)
print("Upper bounds:", upper_bounds)

Lower bounds: [0.239181 0.848233 0.97     0.97    ]
Upper bounds: [0.479181 1.       1.       1.      ]


In [10]:
kappa = 0.5

def negative_ucb(point):
    point = np.asarray(point).reshape(1, -1)

    mean_scaled, std_scaled = gp.predict(
        point,
        return_std=True
    )

    ucb_scaled = mean_scaled[0] + kappa * std_scaled[0]

    return -ucb_scaled

In [11]:
from scipy.optimize import minimize

rng = np.random.default_rng(42)

starting_points = [
    best_x,
    week1_x[0],
    week3_x[0]
]

random_starts = rng.uniform(
    lower_bounds,
    upper_bounds,
    size=(70, 4)
)

starting_points.extend(random_starts)

results = []

for start in starting_points:
    start = np.clip(start, lower_bounds, upper_bounds)

    result = minimize(
        negative_ucb,
        x0=start,
        method="L-BFGS-B",
        bounds=bounds
    )

    if result.success:
        results.append(result)

if not results:
    raise RuntimeError("No successful optimisation runs")

best_result = min(results, key=lambda result: result.fun)
query = best_result.x

In [12]:
query_mean_scaled, query_std_scaled = gp.predict(
    query.reshape(1, -1),
    return_std=True
)

query_mean = query_mean_scaled[0] * Y_std + Y_mean
query_std = query_std_scaled[0] * Y_std
query_ucb = query_mean + kappa * query_std

nearest_distance = np.min(
    np.linalg.norm(X - query, axis=1)
)

print("Method: Boundary-aware trust-region GP-UCB")
print("Suggested Week 4 query:", query)

print(
    "Portal format:",
    "-".join(f"{value:.6f}" for value in query)
)

print("Predicted mean:", query_mean)
print("Predicted std:", query_std)
print("UCB:", query_ucb)
print("Distance to nearest observation:", nearest_distance)

Method: Boundary-aware trust-region GP-UCB
Suggested Week 4 query: [0.479181 1.       1.       1.      ]
Portal format: 0.479181-1.000000-1.000000-1.000000
Predicted mean: 4279.929452797422
Predicted std: 218.55888554681485
UCB: 4389.208895570829
Distance to nearest observation: 0.13982311071135559


In [13]:
matern_kernel = gp.kernel_.k1.k2
lengthscales = np.asarray(matern_kernel.length_scale)

importance = 1 / lengthscales
importance = importance / importance.sum()

print("Lengthscales:", lengthscales)

for i, value in enumerate(importance, start=1):
    print(f"x{i}: {value}")

Lengthscales: [2.         0.29852272 0.37737104 2.        ]
x1: 0.07143121717291787
x2: 0.4785647004073676
x3: 0.3785728652467967
x4: 0.07143121717291787
